## Open Exploration

*Kleine intro over wat er in dit bestand gebeurd*

In [2]:
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt


In [3]:
# make sure pandas is version 1.0 or higher
# make sure networkx is version 2.4 or higher
print(pd.__version__)
print(nx.__version__)

2.2.2
3.2.1


In [4]:
from ema_workbench import (Model, Policy, ema_logging, SequentialEvaluator, MultiprocessingEvaluator, perform_experiments, save_results, load_results)

from problem_formulation import get_model_for_problem_formulation, sum_over, sum_over_time

In [5]:
ema_logging.log_to_stderr(ema_logging.INFO)

# choose problem formulation number, between 0-5
# each problem formulation has its own list of outcomes
dike_model, planning_steps = get_model_for_problem_formulation(2)

In [6]:
def get_do_nothing_dict():
    return {l.name:0 for l in dike_model.levers}

# Define 4 policies
policies = [
    # This is the base case or business as usual policy (BAU), where nothing is done. 
    #As analysts we want to have a base case, so that we can compare the results of the other policies with this one.
    
    Policy("Base case", **{l.name: 0 for l in dike_model.levers}),

    # Increase dike height with 5 dm every decision step
    #This is a policy that is our worst case scenario, where we do not implement any RfR and only increase the dike height.
    Policy("Dikes only", **dict(get_do_nothing_dict(), **{
        'A.1_DikeIncrease 0': 5, 'A.1_DikeIncrease 1': 5, 'A.1_DikeIncrease 2': 5,
        'A.2_DikeIncrease 0': 5, 'A.2_DikeIncrease 1': 5, 'A.2_DikeIncrease 2': 5,
        'A.3_DikeIncrease 0': 5, 'A.3_DikeIncrease 1': 5, 'A.3_DikeIncrease 2': 5,
        'A.4_DikeIncrease 0': 5, 'A.4_DikeIncrease 1': 5, 'A.4_DikeIncrease 2': 5,
        'A.5_DikeIncrease 0': 5, 'A.5_DikeIncrease 1': 5, 'A.5_DikeIncrease 2': 5
    })),

    # Implement RfR on all locations at all decision steps, this is the policy that we see as the best possible option
    #As analysts we want to have as much RfR possible, so that why we set all RfR levers to 1.
    Policy("RfR only", **dict(get_do_nothing_dict(), **{
        '0_RfR 0': 1, '0_RfR 1': 1, '0_RfR 2': 1,
        '1_RfR 0': 1, '1_RfR 1': 1, '1_RfR 2': 1,
        '2_RfR 0': 1, '2_RfR 1': 1, '2_RfR 2': 1,
        '3_RfR 0': 1, '3_RfR 1': 1, '3_RfR 2': 1,
        '4_RfR 0': 1, '4_RfR 1': 1, '4_RfR 2': 1,
        '5_RfR 0': 1, '5_RfR 1': 1, '5_RfR 2': 1,
        '6_RfR 0': 1, '6_RfR 1': 1, '6_RfR 2': 1
    })),

    #This is the final policy proposed by Rijkswaterstaat and that was approved by the stakeholders in the second round of debates.
     Policy("Final Policy", **dict(get_do_nothing_dict(),
                                      **{'0_RfR 0':1,
                                         '2_RfR 0':1,
                                         '3_RfR 0':1,
                                         'EWS_DaysToThreat': 3,
                                         'A.3_DikeIncrease 0':0.5,
                                          'A.4_DikeIncrease 0':0.5,
                                          'A.5_DikeIncrease 0':0.5,
    }))
]

#All these policies are implemented in the model, we will save the results of the scenarios in the following section, so that we can use them 
#in an addapted MORDM. That way with our limited computational resources we can still run diverse scenarios and get interestng ideal policies.


In [7]:
#for policy in policies:
    #print(f"Running policy {policy.name}")
    #with MultiprocessingEvaluator(dike_model) as evaluator:
        #results = perform_experiments(evaluator, 100, policy, planning_steps=planning_steps)

    # Save the results to a file
   # save_results(results, f"results_{policy.name}.tar.gz")
   #Ik heb dit gemaakt maar is niet nodig

In [ ]:
n_scenarios = 1000

# running the model through EMA workbench
with MultiprocessingEvaluator(dike_model) as evaluator:
    experiments, outcomes = evaluator.perform_experiments(n_scenarios, policies)

#save_results((experiments, outcomes), "open_exploration.tar.gz")

In [ ]:
experiments, outcomes = load_results('open_exploration.tar.gz')

policies = experiments['policy']

data = pd.DataFrame.from_dict(outcomes)
data['policy'] = policies

plot = sns.pairplot(data, hue='policy', vars=outcomes.keys(), )
plot.fig.suptitle("Pair Plot of Open Exploration Results", fontsize=16)
plot.fig.subplots_adjust(top=0.95)
plot.savefig("open_exploration.png")
plt.show()

Extract the values of the uncertainties of scenario that describe a best case and a worst case. These scenarios are saved and later used as reference scenarios in the Multi-scenario MORDM.

In [10]:
uncertainties = [unc.name for unc in dike_model.uncertainties]

# Unpack results
experiments, outcomes = load_results('open_exploration.tar.gz')

# Convert 'Expected Number of Deaths' outcome into a DataFrame
deaths = pd.DataFrame(outcomes['Expected Number of Deaths'])


[MainProcess/INFO] results loaded successfully from /Users/evabrouwer/Desktop/EPA141A_Group12_2025/final assignment/open_exploration.tar.gz


In [11]:
# Filter for only "RfR only" policy
rfr_only_mask = experiments['policy'] == 'RfR only'
deaths_rfr = deaths[rfr_only_mask].copy()

# Add scenario and policy info
deaths_rfr['scenario'] = experiments[rfr_only_mask].index
deaths_rfr['policy'] = experiments[rfr_only_mask]['policy']

# Sum deaths across locations/years (if multi-dimensional)
deaths_rfr['total_deaths'] = deaths_rfr.drop(columns=['scenario', 'policy']).sum(axis=1)

# Find the scenario with the minimum deaths for RfR only - this is because we want to minimize the number of deaths.
min_idx = deaths_rfr['total_deaths'].idxmin()
min_experiment = experiments.loc[min_idx]

print("Lowest number of deaths (RfR only):")
print(f"Policy: {min_experiment['policy']}, Scenario index: {min_idx}")
print(f"Total deaths: {deaths_rfr.loc[min_idx]['total_deaths']}")
print("\nLever settings for lowest death scenario:")
print(min_experiment[uncertainties])

min_uncertainties = min_experiment[uncertainties]
min_uncertainties_df = pd.DataFrame([min_uncertainties])

import pickle

with open("reference_uncertainties_rfr_only.pkl", "wb") as f:
    pickle.dump(min_uncertainties_df, f)

Lowest number of deaths (RfR only):
Policy: RfR only, Scenario index: 2098
Total deaths: 0.0

Lever settings for lowest death scenario:
discount rate 0                   4.5
discount rate 1                   3.5
discount rate 2                   1.5
A.0_ID flood wave shape           113
A.1_Bmax                   141.175158
A.1_pfail                    0.676314
A.1_Brate                         1.0
A.2_Bmax                   288.594708
A.2_pfail                    0.913399
A.2_Brate                         1.5
A.3_Bmax                    205.96233
A.3_pfail                     0.69333
A.3_Brate                         1.5
A.4_Bmax                   140.241782
A.4_pfail                    0.662488
A.4_Brate                         1.5
A.5_Bmax                   344.506533
A.5_pfail                    0.864793
A.5_Brate                        10.0
Name: 2098, dtype: object


In [12]:
# Filter for only "Dike only" policy
dikes_only_mask = experiments['policy'] == 'Dikes only'
deaths_dikes = deaths[dikes_only_mask].copy()

# Add scenario and policy info
deaths_dikes['scenario'] = experiments[dikes_only_mask].index
deaths_dikes['policy'] = experiments[dikes_only_mask]['policy']

# Sum deaths across locations/years (if multi-dimensional)
deaths_dikes['total_deaths'] = deaths_dikes.drop(columns=['scenario', 'policy']).sum(axis=1)

# Find the scenario with the minimum deaths for dike only - this is because we want to minimize the number of deaths.
max_idx = deaths_dikes['total_deaths'].idxmax()
max_experiment = experiments.loc[max_idx]

print("Highest number of deaths (Dikes only):")
print(f"Policy: {max_experiment['policy']}, Scenario index: {max_idx}")
print(f"Total deaths: {deaths_dikes.loc[max_idx]['total_deaths']}")
print("\nLever settings for highest death scenario:")
print(max_experiment[uncertainties])

max_uncertainties = max_experiment[uncertainties]
max_uncertainties_df = pd.DataFrame([max_uncertainties])

with open("reference_uncertainties_dikes_only.pkl", "wb") as f:
    pickle.dump(max_uncertainties_df, f)

Highest number of deaths (Dikes only):
Policy: Dikes only, Scenario index: 1936
Total deaths: 1.07639502455687

Lever settings for highest death scenario:
discount rate 0                   3.5
discount rate 1                   1.5
discount rate 2                   1.5
A.0_ID flood wave shape            56
A.1_Bmax                    59.581609
A.1_pfail                     0.89347
A.1_Brate                        10.0
A.2_Bmax                    32.743916
A.2_pfail                     0.98342
A.2_Brate                        10.0
A.3_Bmax                   257.255803
A.3_pfail                     0.00035
A.3_Brate                         1.0
A.4_Bmax                   324.002181
A.4_pfail                    0.186928
A.4_Brate                         1.0
A.5_Bmax                   327.282634
A.5_pfail                    0.126277
A.5_Brate                        10.0
Name: 1936, dtype: object
